In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of a fictional location called Lunapolis, including its capital, weather, population of cheese miners, and potential labor union actions.\n\n## SUMMARY\n- The capital of the moon is Lunapolis.\n- The weather in Lunapolis is described as clear skies with a high of 120°C and a low of -100°C.\n- There are 100,000 cheese miners residing in Lunapolis.\n- A potential strike by the cheese miners' union is anticipated due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='850b801a-8ce7-4d0c-a223-06240ba49738'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='e0805465-369b-4c09-9c31-3541e9d5f3ed'),
             

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of a fictional location called Lunapolis, including its capital, weather, population of cheese miners, and potential labor union actions.

## SUMMARY
- The capital of the moon is Lunapolis.
- The weather in Lunapolis is described as clear skies with a high of 120°C and a low of -100°C.
- There are 100,000 cheese miners residing in Lunapolis.
- A potential strike by the cheese miners' union is anticipated due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [13]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [14]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [15]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='5df33064-f7d9-4111-aec3-7e720e2cdaec'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='26fadf34-bdd1-49ac-94e4-ea2eb1f9b262', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='0dcb3658-3094-49a6-8347-a9119eac79a8'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='1bf8a77b-fa91-4ce6-bac4-903c7b6b4e0f', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='7688d912-f835-471a-94e2-40dba478c430'),
              AIMessage(content='I can’t read the device’s temperature from here. If you want t

In [9]:
print(response["messages"][-1].content)

I can’t read the device’s internal temperature from here. If you want to check it yourself, please tell me what kind of device this is (laptop, desktop, smartphone, tablet) and the model/OS. In the meantime, here’s how to handle temperature issues and checking temps once you can power it:

- If the device is hot to the touch and won’t turn on:
  - Unplug it and let it cool on a hard, flat surface for 15–30 minutes.
  - Do not try to power it on while it’s hot.
  - If it’s still hot after cooling, check vents for dust and ensure it sits on a surface that doesn’t block airflow. Try a different outlet or charger once cooled.

- If you can power it on and want to check temperatures:
  - Windows: use a hardware monitor tool such as HWiNFO or Core Temp.
  - macOS: tools like iStat Menus or Macs Fan Control (third-party).
  - Linux: run lm-sensors (sensors command) to read CPU/GPU temps.
  Typical healthy ranges:
  - Idle CPU: roughly 30–50°C
  - Under load CPU/GPU: often 60–85°C (can go high